# 固定 beta 的配对价差：从两条腿到全局对冲比率

本教案讨论统计套利中的一个明确选择：**beta 固定，不在持仓期间频繁调整**。我们将两条日线组成 spread，并用端点去趋势后的残差平方，衡量一个研究期内最平稳的全局 beta。

配套黑板：`rolling_mean_derivative_blackboard.nb`。黑板负责读取 `quant_data.metapairs`、选择 pair、画图和交互；本 Notebook 负责把数学定义说清楚。

## 1. 两条腿必须先对齐

从 `quant_data.us_market_prices` 读取日线时，先取两只股票在同一交易日都有价格的共同日期。设共同日线的**原始价格**为 $Y_t$ 与 $X_t$。

建模不做首日价格归一化：每个 $Y_t,X_t$ 都是该交易日的原始 price。这样 beta 保持在实际价格单位下的对冲比率，也不会因为改变样本起点而仅因重设归一化基准改变尺度。

## 2. 固定 beta 与 spread

指定一个常数 $\beta$ 后：

$$S_t(\beta)=Y_t-\beta X_t$$

- 做多 spread：多 $1$ 单位 $Y$、空 $\beta$ 单位 $X$。
- 做空 spread：空 $1$ 单位 $Y$、多 $\beta$ 单位 $X$。

这里的单位是原始价格腿的研究单位。beta 是每 $1$ 单位 $Y$ 对应的 $X$ 价格单位对冲比率；实际下单前仍应结合可交易股数、美元名义金额与风险敞口确定仓位。

## 3. 为什么 beta 改变时 spread 的均线会上下移动？

若两条腿的中心高度约为 $B_Y,B_X$，则：

$$\mathbb{E}[S]\approx B_Y-\beta B_X$$

所以 beta 同时影响两件事：

1. 两条腿的波动抵消程度；
2. spread 所处的基线高度。

因此，白色 trailing mean 的上下移动不是错误；它是固定 beta 改变 spread 水平的直接结果。

## 4. 不用 rolling beta，怎样选择一个全局 beta？

对每一个候选 beta，先计算整个训练区间的 spread。再连接它的第一个点和最后一个点，得到端点趋势线：

$$L_t(\beta)=S_1(\beta)+\frac{t-1}{T-1}\left[S_T(\beta)-S_1(\beta)\right]$$

端点去趋势残差为：

$$r_t(\beta)=S_t(\beta)-L_t(\beta)$$

选择使残差平方均值最小的 beta：

$$\beta^*=\arg\min_\beta\;\frac{1}{T}\sum_{t=1}^{T}r_t(\beta)^2$$

## 5. 为什么可以直接算，而不必逐个试 beta？

令 $D(v)$ 表示从序列 $v$ 中减去其两端连线。端点去趋势是线性操作，因此：

$$r(\beta)=D(Y)-\beta D(X)$$

最小二乘的一阶条件给出：

$$\boxed{\beta^*=\frac{\sum_tD(X)_tD(Y)_t}{\sum_tD(X)_t^2}}$$

这就是黑板按钮 `fit endpoint-detrended global beta` 所计算的值。浅灰虚线是 $L_t(\beta)$，不是 rolling mean；白线才是 trailing rolling mean。

## 6. AA/CENX 的当前研究结果

以 2022-03-31 至 2026-08-07 的共同日线的原始 price 计算，端点去趋势的全样本研究值应按上式重新拟合。此前基于首日归一化得到的 $0.4462$ 不适用于原始价格单位。

这不是未来可直接保证的实盘 beta。它回答的是：**在这段完整历史中，哪个固定 beta 让端点去趋势后的 spread 最平稳？**

## 7. 无未来函数的正确使用方式

trailing mean 本身在每一天都没有未来函数：

$$M_t=\frac1w\sum_{k=0}^{w-1}S_{t-k}$$

但若用整段历史（包括未来日期）拟合 $\beta^*$，再回头评价早期信号，就有前视偏差。正确流程应是：

1. 选定训练截止日；
2. 只用截止日及之前的数据拟合固定 beta；
3. 冻结 beta；
4. 在之后的独立测试/实盘区间计算 trailing mean、标准差和交易信号。

beta 可以固定很久；“不滚动”不等于“可以用未来数据拟合”。

## 8. sigma 与 ±2 sigma 带怎样计算？它假设正态分布吗？

固定 $\beta$ 后，先得到每个时点的 spread $S_t=Y_t-\beta X_t$。在时点 $t$，只取当前及之前的最近 $w$ 个 spread：

$$M_t=\frac{1}{w}\sum_{k=0}^{w-1}S_{t-k}$$

$$\sigma_t=\sqrt{\frac{1}{w}\sum_{k=0}^{w-1}(S_{t-k}-M_t)^2}$$

### 把符号翻成一句一句的话

下标 $t$ 表示“现在这根 bar”；$t-1$ 是上一根，$t-2$ 是上两根。因此 $S_{t-k}$ 只是“从现在往回数第 $k$ 根的 spread”。求和符号 $\sum_{k=0}^{w-1}$ 是一个简写：把窗口里的 $w$ 个数逐一相加。

- $M_t$：把 $S_t,S_{t-1},\ldots,S_{t-w+1}$ 相加，再除以 $w$。它就是这段近期 spread 的中心值，或“最近的常态水平”。
- $S_{t-k}-M_t$：窗口中某一个 spread 距离该中心值多远；正数表示在均线上方，负数表示在下方。
- $(S_{t-k}-M_t)^2$：先平方，令上方和下方的偏离都成为正数，避免 $+3$ 与 $-3$ 相加后错误地抵消为零。
- $\sigma_t$：把这些平方偏离求平均后开平方。开平方把单位从“价格平方”变回 spread 的原始价格单位，所以 sigma 可以直接与 spread 相加、相减。

本项目的 Simple Dashboard 有两种用途不同的 window，不能混为一谈：执行层的 rolling mean、sigma 与 z-score 使用 20 个交易日 $\times$ 每日 79 根 5 分钟 bar，即 $w=20\times79=1580$；$20\times77$ 是 kappa 估计专用的窗口。图上的红色 spread band 为了便于阅读，使用每日最后一根 bar 组成的 20 个日收盘 spread，即图形 overlay 的 $w=20$。

当前实现使用分母 $w$（`ddof=0`）：它把这一个 rolling window 当作描述当前局部波动的总体；不是分母为 $w-1$ 的无偏样本标准差。然后直接画出：

$$U_t=M_t+2\sigma_t,\qquad L_t=M_t-2\sigma_t$$

所以 $\pm2\sigma$ 首先只是由最近 window 内实际 spread 算出的**波动尺度带**，计算本身完全不需要正态分布假设。对应的标准化距离是：

$$z_t=\frac{S_t-M_t}{\sigma_t}$$

### 为什么正态近似会给出约 95%？

先把“约 95%”和 sigma 的计算分开：sigma 的计算不需要任何分布假设；95% 来自随后对标准化变量 $z$ 作的正态近似。若在一个局部稳定区间内，真实的中心为 $\mu$、真实波动为 $\sigma$，并且：

$$Z=\frac{S-\mu}{\sigma}\sim N(0,1)$$

那么“spread 距中心不超过两个 sigma”就是 $-2\le Z\le2$。标准正态分布的累计概率为：

$$P(\mu-2\sigma\le S\le\mu+2\sigma)=P(-2\le Z\le2)=\Phi(2)-\Phi(-2)\approx0.9545$$

也就是说，若上述模型真的成立，长期重复观察时约 95.45% 的点会落在带内，约 4.55% 会落在带外；常用的 95% 正态分位则是 $\pm1.96\sigma$。这不是“当前这一点在带外后有 95% 概率回归”的含义。

实际图中用的是同一 rolling window 估出的 $M_t$ 和 $\sigma_t$，而非未知的真实 $\mu,\sigma$；金融 spread 还常有自相关、波动聚集、厚尾、跳空，以及 beta/均值/标准差的估计误差。因此红带不是严格的 95% 置信区间，也不是触及后必然均值回归的保证。策略中更稳妥的含义是：它把当前偏离按近期波动标准化，供 entry、风险控制和不同 pair 的可比性使用。

## 9. 黑板操作流程

1. 点击 **从 metapairs 读取 pair 列表**。
2. 从下拉菜单选择例如 `aa_cenx`、`jpm_bac` 或 `pfe_mrk`。
3. 点击 **加载所选 pair 的共同日线**。
4. 手动输入固定 beta，或点击全样本研究拟合按钮。
5. 调节 `spread mean window`，观察白色 trailing mean、红色 trailing ±2σ 带与浅灰端点趋势线。

下一课可以加入训练截止日控制，让固定 beta 的拟合与 out-of-sample 回测严格分离。

## 10. 练习

- 为什么 $\beta=0$ 时 spread 就是 $Y$？为什么 beta 很大时图形会主要像 $-X$？
- 在 `aa_cenx` 中比较两个手动设定的 beta 与按原始 price 拟合得到的 $\beta^*$ 的端点趋势残差。
- 选择另一个 metapair。它的全局 beta 是否仍小于 1？这能否直接解释为应该多买某条腿？为什么不能？
- 将训练样本限定在较早区间，再检查冻结 beta 在较晚区间是否仍让 spread 平稳。
- 对一个 $w=20$ 的 window，分别用 $w$ 与 $w-1$ 作分母计算 sigma；两条 ±2 sigma 带有什么差别？这种差别是否表示行情本身发生变化？
- 为什么“落在 ±2 sigma 外”不能单独证明下一根 bar 会回到均线？列出至少两个本节提到的原因。